# Byte-Pair Encoding (BPE)

**难度:** Hard

实现字节对编码（BPE）分词。

BPE 通过迭代合并最频繁的相邻符号对来构建子词词表，广泛用于现代语言模型。

**签名:** `SimpleBPE()`（类）

**方法:**
- `train(corpus: list[str], num_merges: int)` — 从词列表中学习合并规则
- `encode(text: str) -> list[str]` — 将文本分词为子词 token

**约束:**
- 使用 `</w>` 标记词边界
- `self.merges` 按顺序存储学习到的合并对
- `encode` 按顺序应用合并规则

**提示:**
train：每个词拆分为字符 + `</w>`，统计相邻对频率，合并最高频对，重复
encode：按学习到的合并顺序将文本拆分为子词 token

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

class SimpleBPE:
    def __init__(self):
        self.merges = []  # 存储学习到的合并规则，按学习顺序

    # ---- Step 1: 训练 ----
    def train(self, corpus, num_merges):
        # 初始化词表：每个词拆分为字符元组 + '</w>' 标记
        # 例如 "low" → ('l', 'o', 'w', '</w>')
        # 用元组是因为它可以作为字典的 key（list 不行）
        vocab = {}
        for word in corpus:
            symbols = tuple(word) + ('</w>',)
            vocab[symbols] = vocab.get(symbols, 0) + 1  # 记录词频

        self.merges = []

        # 迭代合并 num_merges 次
        for _ in range(num_merges):
            # ---- Step 1a: 统计所有相邻对的频率 ----
            pairs = {}
            for word, freq in vocab.items():
                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    # 频率累加：同一个 pair 可能出现在多个词中
                    pairs[pair] = pairs.get(pair, 0) + freq

            if not pairs:
                break  # 没有可合并的 pair 了

            # ---- Step 1b: 选择最高频的 pair ----
            best = max(pairs, key=pairs.get)
            self.merges.append(best)

            # ---- Step 1c: 在所有词中执行合并 ----
            new_vocab = {}
            for word, freq in vocab.items():
                new_word = []
                i = 0
                while i < len(word):
                    # 如果当前位置匹配 best pair，合并
                    if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                        new_word.append(word[i] + word[i + 1])
                        i += 2  # 跳过两个字符
                    else:
                        new_word.append(word[i])
                        i += 1
                new_vocab[tuple(new_word)] = freq
            vocab = new_vocab

    # ---- Step 2: 编码 ----
    def encode(self, text):
        all_tokens = []
        for word in text.split():
            # 拆分为字符 + '</w>'
            symbols = list(word) + ['</w>']
            # 按训练时的合并顺序，依次应用每个合并规则
            for a, b in self.merges:
                i = 0
                while i < len(symbols) - 1:
                    if symbols[i] == a and symbols[i + 1] == b:
                        # 合并：替换两个元素为一个
                        symbols = symbols[:i] + [a + b] + symbols[i + 2:]
                    else:
                        i += 1
            all_tokens.extend(symbols)
        return all_tokens

In [ ]:
# Demo
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges)
print('Encode:', bpe.encode('low lower newest'))

In [ ]:
from torch_judge import check
check('bpe')

## 📝 核心思路总结

1. **BPE 的核心：贪心合并最高频相邻对**：从字符级开始，迭代合并最频繁的 pair
2. **`</w>` 标记词边界**：区分词尾字符和独立字符（如 "at" vs "at</w>"）
3. **encode 按训练顺序应用合并规则**：顺序很重要，先学的合并优先级更高